In [1]:
import os
from trame.app.demo import Cone

print(os.environ.get("TRAME_JUPYTER_WWW"))

app = Cone()
await app.ui.ready

print(app.ui._jupyter_content())

app.ui

None
<iframe id="trame_trame__template_main" src="http://localhost:51013/index.html?ui=main&reconnect=auto" style="border: none; width: 100%; height: 600px;"></iframe>


In [4]:
from trame.app import get_server
from trame.widgets import html
from trame.ui.html import DivLayout

server = get_server()
state = server.state

def reset_slider():
    state.slider_a = 1

@state.change("slider_a")
def udpate_result(slider_a, **_):
    state.result = slider_a / 2

with DivLayout(server, 'a', height=30) as ui_a:
    html.Input(
        type="range", 
        min=-1, 
        max=50, 
        step=0.1, 
        v_model_number=("slider_a", 2), 
        style="width: 100%;",
    )

await ui_a.ready
ui_a

In [5]:
with DivLayout(server, 'b', height=30) as ui_b:
    html.Button(
        "Reset Value", 
        click=reset_slider,
    )
    html.Span("{{ slider_a }} / 2 = {{ result }}", style="margin-left: 2rem")

ui_b

In [6]:
from pyvcell.sim_results.widgets import App

In [7]:
app = App()
await app.ui.ready 
app.ui

In [5]:
import pyvista as pv
sphere = pv.Sphere()

# long example
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(sphere)
plotter.show(jupyter_backend='trame')

Widget(value='<iframe src="http://localhost:52657/index.html?ui=P_0x16a489b70_5&reconnect=auto" class="pyvista…

In [1]:
import pyvista as pv
from pyvista.trame.ui import plotter_ui
from trame.app import get_server
from trame.ui.vuetify3 import SinglePageLayout
from trame.widgets import vuetify3
from vtkmodules.vtkFiltersSources import vtkConeSource

import pyvista 
from pathlib import Path 
from pyvista import DataSet 


def read_mesh(path: str | None = None) -> DataSet:
    new_mesh_file = Path(path or "/Users/alex/Desktop/uchc_work/repos/pyvcell/examples/test_output/vox8.vtu")
    pyvista_mesh = pyvista.read(str(new_mesh_file))
    return pyvista_mesh


def update_clip(pyvista_mesh: DataSet, clip_level: float):
    bounds = pyvista_mesh.bounds
    clip_position = bounds[0] + clip_level * (bounds[1] - bounds[0])  
    
    clipped = pyvista_mesh.clip_box(bounds=(clip_position,) + bounds[1:])  

    pl.clear()  
    pl.add_mesh(clipped, show_scalar_bar=True)  

    pl.show_bounds()  
    # pl.show_grid()  

    pl.render()  
    ctrl.view_update() 


pv.OFF_SCREEN = True

server = get_server()
state, ctrl = server.state, server.controller

source = read_mesh()  # Load an unstructured grid
pl = pv.Plotter(notebook=True)
pl.add_mesh(source, color="seagreen")


@state.change("clip_level")
def update_clipping(clip_level, **kwargs):
    """Reactively update the clip level when slider is moved"""
    update_clip(source, clip_level)


async def run():
    with SinglePageLayout(server) as layout:
        with layout.toolbar:
            vuetify3.VSpacer()
            
            # Clipping slider (now properly updates dynamically)
            vuetify3.VSlider(
                v_model=("clip_level", 0.5),  # Default midpoint clip level
                min=0.0,
                max=1.0,
                step=0.05,
                hide_details=True,
                density="compact",
                style="max-width: 300px",
            )
        
            vuetify3.VProgressLinear(
                indeterminate=True,
                absolute=True,
                bottom=True,
                active=("trame__busy",),
            )
    
        with (
            layout.content,
            vuetify3.VContainer(
                fluid=True,
                classes="pa-0 fill-height",
            ),
        ):
            # Use PyVista UI template for Plotters
            view = plotter_ui(pl)
            print(type(view))
            ctrl.view_update = view.update
    
    # Show UI
    await layout.ready
    return layout


In [2]:
await run()

<class 'pyvista.trame.views.PyVistaRemoteLocalView'>


HTML(value='<iframe id="trame_trame__template_main" src="http://localhost:54695/index.html?ui=main&reconnect=a…